# Transformer AR vs causal encoder — last checkpoint, singletons and longer instructions (8 runs)

| condition | slots S | train k | held-out tests |
|---|---|---|---|
| S5_k0234 | 5 | 0,2,3,4 | k=1 at every slot; k=5 (243 words) |
| S6_k02345 | 6 | 0,2,3,4,5 | k=1; k=6 (729 words) |
| S7_k02345 | 7 | 0,2,3,4,5 | k=1; k=6 (729); k=7 (2187 words) |
| S7_k023456 | 7 | 0,2,3,4,5,6 | k=1; k=7 (2187 words) |

**Format.** Visible `<identity>` fillers, learned positions. Every longer test stays within S, so it adds no new
physical positions.

**Models.** `transformer_ar` (autoregressive, free-running) and `causal_encoder` (causal attention with 36 parallel
output queries, no feedback).

**Training.** seed 42, 5000 × 256.
* One common initialization per architecture: all conditions share positional capacity 7.
* S6_k02345 and S7_k02345 see the identical logical training stream.

**Evaluation: the LAST checkpoint, i.e. the model after update 5000.** The best checkpoint is still saved, and its
step is reported for reference.

**Test size.** 1000 test grids, except k=7, which uses `K7_GRIDS` grids (default 300, i.e. 656,100 records per run).

**Send back `LongK_AR_vs_CausalEnc_Review.ipynb`** (printed and downloaded by the last cell).


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_CONDITIONS = ["S5_k0234", "S6_k02345", "S7_k02345", "S7_k023456"]
RUN_MODELS = ["transformer_ar", "causal_encoder"]
N_TEST_GRIDS = 1000
K7_GRIDS = 300                 # grids used for the 2187 k=7 words (raise to 1000 for the full set; ~3x the time)
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
for e in RUN_CONDITIONS:
    print(e, "expected test records per model:", f"{LK.expected_records(e, N_TEST_GRIDS, K7_GRIDS):,}")
print("results ->", LX.family_root(RESULTS_ROOT, MODE))


In [ ]:
# --- Train + evaluate the LAST checkpoint, sequentially (resumable) --------------------------------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR),
                      n_grids=N_TEST_GRIDS, k7_grids=K7_GRIDS, device=DEVICE)
display(status)


In [ ]:
# --- Report + review export (reads saved artifacts only) -----------------------------------------------------------
rp = LX.report(RESULTS_ROOT, MODE, SEED)
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp))
